# 01 · 데이터 불러오기와 정제  (모듈 2)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/amnotyoung/oda-ai-stats/blob/main/notebooks/01_load_clean.ipynb)

> **World Bank 개발지표(WDI) 패널**을 GitHub에서 바로 불러와 구조를 보고 정제한다.
> 국가×연도 데이터(2000~2022). 오늘의 원칙: **AI가 코드를 짜고 → 우리가 검증한다.**

지표: 1인당 GDP, 기대수명, 5세 미만 사망률, 인구, 초등 취학률 + 지역·소득그룹.

## 0) 불러오기 — GitHub에서 바로 (설치 불필요)

In [ ]:
import pandas as pd, numpy as np
RAW = "https://raw.githubusercontent.com/amnotyoung/oda-ai-stats/main/data/wdi_panel.csv"
df = pd.read_csv(RAW)
print("행 x 열:", df.shape)
df.head()

## 1) 구조 파악 — 컬럼·자료형·결측
진짜 데이터는 **결측이 있다**. 어디에 얼마나 빠졌는지 먼저 본다.

In [ ]:
df.info()

In [ ]:
df.isna().sum()   # 컬럼별 결측 — under5_mort·prim_enroll에 실제 결측 존재

### 🔎 검증 포인트 — 단위·범위·상식
- 기대수명은 0~90세 범위인가? · 1인당 GDP는 양수인가? · 결측을 어떻게 다룰까?

In [ ]:
print("기대수명 범위:", df.life_exp.min(), "~", df.life_exp.max())
print("1인당 GDP 음수:", (df.gdp_pc <= 0).sum(), "건")
df[["gdp_pc","life_exp","under5_mort","pop","prim_enroll"]].describe().round(1)

## 2) 정제 + 파생변수
분석에 쓸 핵심 변수의 결측 행을 제거하고, 금액·인구는 **로그변환**(왜도 큼).

In [ ]:
work = df.dropna(subset=["gdp_pc","life_exp"]).copy()
work["log_gdp"] = np.log(work["gdp_pc"])
work["log_pop"] = np.log(work["pop"])
print("정제 후 행:", len(work), "/ 원본:", len(df))
work.head(3)

## 3) 빠른 요약 — 소득그룹별 평균 기대수명

In [ ]:
work.groupby("income_name")["life_exp"]\
    .agg(국가연도수="count", 평균기대수명="mean")\
    .round(1).sort_values("평균기대수명")

---
✅ **정리**: 불러오기 → 구조·결측 파악 → 검증 → 정제 → 요약. AI에게 시켜도 **이 흐름과 검증은 사람 몫**.

🔁 **폐쇄망 STATA로는?** 같은 작업을 `stata/01_load_clean.do` 에.